## 1. Import Libraries, Configuration, and Utilities
Here we import libraries, and define constants used
throughout the notebook. We also define utility functions for model.

### 1.1. Import Libraries

In [1]:
# Import necessary libraries
import re
import string

import emoji
import matplotlib.pyplot as plt
# NLP and preprocessing
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import Dataset
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
# Machine learning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, hamming_loss, f1_score, jaccard_score)
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
# Deep learning
import torch
from torch import nn
import  transformers
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)

/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2. Configuration Parameters

In [2]:
TEST_DATA_PATH = 'data/track-a-test.csv'

In [3]:
# Emotion labels we're predicting
EMOTIONS = ['anger', 'fear', 'joy', 'sadness', 'surprise']

In [4]:
# Define model paths and test data path
MODEL_PATHS = [
    './results/roberta-base_vanilla_out',
    './results/bert-base-uncased_vanilla_out',
    './results/distilbert-base-uncased_vanilla_out',
    './results/twitter-roberta-base-emotion-multilabel-latest_vanilla_out'
]

### 1.3. Utility Functions


In [5]:
def compute_metrics(y_true, y_pred, threshold=0.5, verbose=False):
    """
    Compute comprehensive metrics for multi-label classification

    Parameters:
        y_true (ndarray): Ground truth labels
        y_pred (ndarray): Predicted probabilities or binary predictions
        threshold (float): Threshold for converting probabilities to binary predictions
        verbose (bool): Whether to print detailed classification report

    Returns:
        dict: Dictionary containing various evaluation metrics
    """
    # Convert probabilities to binary predictions if needed
    if not isinstance(y_pred[0][0], (np.int64, np.int32, int, bool)):
        y_pred_binary = (y_pred >= threshold).astype(int)
    else:
        y_pred_binary = y_pred

    # Calculate metrics
    metrics = {
        "hamming_loss": hamming_loss(y_true, y_pred_binary),
        "exact_match": np.all(y_true == y_pred_binary, axis=1).mean(),
        "f1_micro": f1_score(y_true, y_pred_binary, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred_binary, average="macro", zero_division=0),
        "jaccard_micro": jaccard_score(y_true, y_pred_binary, average="micro", zero_division=0),
        "jaccard_macro": jaccard_score(y_true, y_pred_binary, average="macro", zero_division=0)
    }

    if verbose:
        print(classification_report(y_true, y_pred_binary, target_names=EMOTIONS, zero_division=0))
        print(f"Exact Match Accuracy: {metrics['exact_match']:.4f}")

    return metrics

## 2. Data Loading & Quick EDA

In [ ]:
# Load the dataset
df = pd.read_csv(TEST_DATA_PATH)

# Display the first few rows to understand the data structure
df.head()

## 3. Evaluate RoBERTa Model

## 4. Model Comparison